# Exploring the AGME Grammar

An interactive notebook for understanding what the AGME model learns.
Demonstrates on a small English corpus (verbal inflections + plural allomorphs)
before scaling to the Brent corpus in `brent_experiment.ipynb`.

**What you can do here:**
1. Train the model on a toy corpus
2. Inspect the learned morpheme lexicon (PYP posterior)
3. Visualize *MAP constraint weights vs. their P-map priors
4. Sample parses and examine phonological mappings
5. Compare MAP vs. sampled segmentations for individual words

## 1. Imports and corpus

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # find agme package

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (10, 4)
matplotlib.rcParams["font.size"] = 11

from agme import Model

In [ ]:
# English verbal/nominal inflection corpus (IPA phoneme strings,
# one character per phoneme).
# Exercises:
#   Voicing assimilation:  /z/ → [s] after voiceless (kæts, bʊks)
#   Epenthesis:            /z/ → [ɪz] after sibilants (bʌsɪz, rozɪz)
#   Past tense:            /d/ → [t] after voiceless (wɔkt, kɪst)
#                          /d/ → [ɪd] after /t,d/ (wɪtɪd)
#   Progressive:           /ɪŋ/ faithful (rʌnɪŋ, sɪŋɪŋ)

CORPUS = [
    # Plural allomorphs
    "dɑgz",   # dogs       UR: /z/ faithful
    "læbz",   # labs       UR: /z/ faithful
    "kæts",   # cats       UR: /z/ → [s] (devoicing)
    "bʊks",   # books      UR: /z/ → [s]
    "bʌsɪz",  # buses      UR: /z/, epenthesis
    "rozɪz",  # roses      UR: /z/, epenthesis

    # Past tense allomorphs
    "hʌgd",   # hugged     UR: /d/ faithful
    "wɔkt",   # walked     UR: /d/ → [t]
    "kɪst",   # kissed     UR: /d/ → [t]
    "wɪtɪd",  # waited     UR: /d/, epenthesis

    # 3sg -s
    "rʌnz",   # runs       UR: /z/ faithful
    "sɪŋz",   # sings      UR: /z/ faithful
    "wɔks",   # walks      UR: /z/ → [s]

    # Progressive -ɪŋ
    "rʌnɪŋ",  # running
    "sɪŋɪŋ",  # singing
    "wɔkɪŋ",  # walking
]

print(f"Corpus size: {len(CORPUS)} words")
print(f"Alphabet inferred from corpus: {sorted(set(''.join(CORPUS)))}")

## 2. Train the model

In [ ]:
# morpheme_classes=["stem", "suffix"] tells the model to look for
# a (possibly absent) prefix-free structure.
# identity_phonology=False means the full *MAP grammar is active.

model = Model(morpheme_classes=["stem", "suffix"])
model.fit(
    CORPUS,
    n_sweeps=300,
    burn_in=100,
    maxent_update_every=10,
    print_every=50,
    seed=42,
)
print("\nTraining complete.")

## 3. Morpheme lexicon (PYP posterior)

In [ ]:
lexicon = model.morphology.morpheme_lexicon()

for cls, counts in lexicon.items():
    total = sum(counts.values())
    print(f"\n--- {cls.upper()} lexicon (total tokens: {total}) ---")
    for ur, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        bar = "█" * int(20 * cnt / max(counts.values()))
        print(f"  {ur:15s}  {cnt:4d}  {bar}")

## 4. Phonological constraint weights

In [ ]:
fw = model.phonology.faithfulness_weights()

# Sort by deviation from prior (large positive deviation = strong learned markedness)
sorted_fw = sorted(fw.items(), key=lambda kv: -abs(kv[1]["deviation"]))

print(f"{'Constraint':<25} {'weight':>8} {'prior':>8} {'deviation':>10}")
print("-" * 55)
for constraint_repr, vals in sorted_fw[:20]:
    dev = vals["deviation"]
    marker = " ◄" if abs(dev) > 0.1 else ""
    print(f"  {constraint_repr:<23} {vals['weight']:>8.3f} {vals['prior']:>8.3f} {dev:>10.3f}{marker}")

In [ ]:
# Scatter plot: learned weight vs. P-map prior
weights = [v["weight"] for v in fw.values()]
priors  = [v["prior"]  for v in fw.values()]

fig, ax = plt.subplots()
ax.scatter(priors, weights, alpha=0.4, s=20)
ax.plot([0, max(priors)], [0, max(priors)], "k--", lw=1, label="w = prior")
ax.set_xlabel("P-map prior (panphon distance)")
ax.set_ylabel("Learned weight")
ax.set_title("*MAP constraint weights vs P-map prior")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Parse individual words

In [ ]:
# MAP parse for each training word
model.print_ur_report()

In [ ]:
# Sample multiple parses for an ambiguous word
word = "bʌsɪz"  # "buses" — should show epenthesis
parses = model.sample_parses(word, n=10)

print(f"10 sampled parses for [{word}]:")
for i, p in enumerate(parses):
    ur = " + ".join(f"{cls}:/{m}/" for cls, m in zip(p.morpheme_classes, p.morphemes))
    print(f"  [{i+1}]  {ur}  →  SR=[{word}]  logP={p.log_prob:.2f}")

In [ ]:
# Surface form types observed in the posterior
model.print_sr_types()

## 6. Predict UR for a novel surface form

In [ ]:
# Given a novel surface form, score candidate URs
novel = "lɪmɪts"
candidates = [
    "lɪmɪt",  # likely stem (limit)
    "lɪm",    # shorter stem
    "lɪmɪtz", # with voiced suffix
]
scored = model.predict(novel, candidates)
print(f"Candidate URs for novel form [{novel}]:")
for ur, lp in scored:
    print(f"  /{ur}/  logP={lp:.3f}")